In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # or ":16:8" (uses less workspace)

In [ ]:
import sys, platform, warnings
from pathlib import Path
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import DataLoader

# 3D / point-cloud libs
import open3d as o3d
try:
    import trimesh  # optional fallback loader; not required for every workflow
    HAS_TRIMESH = True
except Exception:
    HAS_TRIMESH = False
    warnings.warn(
        "trimesh not found; install with `pip install trimesh` if you need a fallback OFF loader."
    )

# ✅ Reproducibility (optional)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 🏷️ Labels & classes (3-way: other / airplane / car)
NUM_CLASSES = 3
CLASS_IDS = {"other": 0, "airplane": 1, "car": 2}
IDX2NAME = {v: k for k, v in CLASS_IDS.items()}

# (Optional) stricter determinism
try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# 🖥️ Environment summary
print("Python          :",
      sys.version.split()[0], f"({platform.system()} {platform.release()})")
print("PyTorch         :", torch.__version__)
print("Open3D          :", o3d.__version__)
print("trimesh         :", "installed" if HAS_TRIMESH else "not installed")

# ⚙️ Device selection
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("\nCUDA available  : True")
    print("Using device    :", device)
    print("GPU count       :", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"  • GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    device = torch.device("cpu")
    print("\nCUDA available  : False")
    print("Using device    :", device)


In [ ]:
from pathlib import Path

# 👇 Change this to your ModelNet40 root
ROOT = Path(
    r"C:\Users\jkarafotis\Desktop\MyDev\Projects\DeepLearning\data\ModelNet40"
)

# 3-way classification: 0=other, 1=airplane, 2=car  (matches CLASS_IDS defined earlier)
# Treat these categories as explicit; everything else becomes "other".
POS_CLASSES = {"airplane", "car"}

# ------------------------------------------------------------
# Training Configuration — WHAT EACH SETTING CONTROLS
# ------------------------------------------------------------

N_POINTS = 1024  
# Number of points sampled per 3D shape.
# Higher = more geometric detail but slower (try 512–2048).
# Common sweet spot for PointNet = 1024.

BATCH_SIZE = 16  
# Number of samples processed in one training step.
# Larger batches need more GPU RAM; smaller batches update more often.

EPOCHS = 1  
# How many complete passes through the training dataset.
# 1 = testing/debug. Real training typically needs 100–200+.

LR = 1e-3  
# Learning rate for optimizer.
# Controls how big each weight update is.
# 1e-3 works well for Adam; higher can diverge, lower trains slowly.

NUM_WORKERS = 0  
# How many parallel workers to use for DataLoader.
# Windows/Jupyter have issues with multiprocessing → keep at 0.
# Linux training servers can safely use 4–8+.

CACHE_IN_RAM = False  
# If True, point clouds are loaded once and stored in RAM.
# Faster training but uses more memory.
# Good for small datasets (<1GB) on systems with 16GB+ RAM.

SAMPLE_METHOD = "uniform"  
# How to sample points from OFF meshes:
#   "uniform" → random surface sampling (fast, widely used)
#   "poisson" → evenly spaced sampling (better quality, slower)


# (Optional) dataset downsizing/balancing knobs for class imbalance
MAX_ITEMS = None           # e.g., 6000 to cap total samples
FRACTION = None            # e.g., 0.5 to keep 50% at random
BALANCE_PER_CLASS = {0: 200, 1: 200, 2: 200}   # e.g., {0: 1000, 1: 1000, 2: 1000}


In [ ]:
import numpy as np
import open3d as o3d
from pathlib import Path


def load_off_robust(path: Path) -> o3d.geometry.TriangleMesh:
    """Parse OFF with tolerant header + polygon faces -> Open3D TriangleMesh."""
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            raw = [
                ln.strip() for ln in f
                if ln.strip() and not ln.lstrip().startswith("#")
            ]

        if not raw or not raw[0].startswith("OFF"):
            raise ValueError(f"Not a valid OFF file: {path}")

        hdr = raw[0]
        i = 1
        tokens = hdr[3:].strip().split()
        if len(tokens) >= 3:
            n_verts, n_faces, _ = map(int, tokens[:3])
        else:
            n_verts, n_faces, _ = map(int, raw[i].split()[:3])
            i += 1

        verts = []
        for _ in range(n_verts):
            x, y, z = map(float, raw[i].split()[:3])
            verts.append([x, y, z])
            i += 1
        verts = np.asarray(verts, dtype=np.float32)

        tris = []
        for _ in range(n_faces):
            parts = raw[i].split()
            i += 1
            k = int(parts[0])
            idxs = list(map(int, parts[1:1 + k]))
            if k >= 3:
                v0 = idxs[0]
                for j in range(1, k - 1):
                    tris.append([v0, idxs[j], idxs[j + 1]])
        if not tris:
            raise ValueError(f"No triangles parsed: {path}")

        mesh = o3d.geometry.TriangleMesh()
        mesh.vertices = o3d.utility.Vector3dVector(verts.astype(np.float64))
        mesh.triangles = o3d.utility.Vector3iVector(
            np.asarray(tris, dtype=np.int32))
        return mesh
    except Exception as e:
        # Fallback to trimesh if available
        if 'HAS_TRIMESH' in globals() and HAS_TRIMESH:
            import trimesh as _tm
            tm = _tm.load(path, file_type='off', force='mesh')
            if not isinstance(tm, _tm.Trimesh):
                tm = _tm.Trimesh(**tm.to_dict())
            o3 = o3d.geometry.TriangleMesh()
            o3.vertices = o3d.utility.Vector3dVector(
                np.asarray(tm.vertices, dtype=np.float64)
            )
            o3.triangles = o3d.utility.Vector3iVector(
                np.asarray(tm.faces, dtype=np.int32)
            )
            return o3
        raise


def sample_points_from_off(path: Path,
                           n_points: int = 1024,
                           method: str = "uniform",
                           normalize: bool = True) -> np.ndarray:
    mesh = load_off_robust(path)
    if method == "poisson":
        pcd = mesh.sample_points_poisson_disk(number_of_points=n_points,
                                              init_factor=5,
                                              use_triangle_normal=True)
    else:
        pcd = mesh.sample_points_uniformly(number_of_points=n_points)
    pts = np.asarray(pcd.points, dtype=np.float32)
    if pts.shape[0] == 0:
        raise ValueError(f"Sampling produced zero points: {path}")
    return normalize_point_cloud(pts) if normalize else pts


def normalize_point_cloud(pts: np.ndarray) -> np.ndarray:
    centroid = pts.mean(axis=0, keepdims=True)
    pts = pts - centroid
    scale = np.max(np.linalg.norm(pts, axis=1))
    if scale > 0:
        pts = pts / scale
    return pts.astype(np.float32)


In [ ]:
import random
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader


class ModelNet40PlaneCarOther(Dataset):
    """
    3-way labels:
       0 → other (any category not airplane or car)
       1 → airplane
       2 → car
       Built-in downsampling:
         - max_items: cap total samples
         - fraction: keep a random fraction (0<frac<=1)
         - balance_per_class: dict {class_idx: keep_count}
    """

    def __init__(self,
                 root: Path,
                 split: str = "train",
                 n_points: int = 1024,
                 cache: bool = False,
                 method: str = "uniform",
                 max_items: int | None = None,
                 fraction: float | None = None,
                 balance_per_class: dict[int, int] | None = None):
        self.root = root
        self.split = split
        self.n_points = n_points
        self.cache = cache
        self.method = method
        self.items = []
        self._cache = {}

        # Build (path, label) list
        categories = sorted([d.name for d in root.iterdir() if d.is_dir()])
        for c in categories:
            cname = c.lower()
            split_dir = root / c / split
            if not split_dir.exists():
                continue
            for off in split_dir.glob("*.off"):
                if cname == "airplane":
                    y = CLASS_IDS["airplane"]  # 1
                elif cname == "car":
                    y = CLASS_IDS["car"]  # 2
                else:
                    y = CLASS_IDS["other"]  # 0
                self.items.append((off, y))

        if not self.items:
            raise RuntimeError(
                f"No .off files found under {root} for split={split}")

        # ----- Downsampling options -----
        if balance_per_class is not None:
            # Cap number of samples per class (e.g., {0: 500, 1: 500, 2: 500})
            buckets = {c: [] for c in CLASS_IDS.values()}  # {0:[],1:[],2:[]}
            # shuffle before bucketing to avoid class-order bias
            tmp = self.items[:]
            random.shuffle(tmp)
            for path, y in tmp:
                if len(buckets[y]) < balance_per_class.get(y, 10**9):
                    buckets[y].append((path, y))
            self.items = [xy for b in buckets.values() for xy in b]
            random.shuffle(self.items)
        elif fraction is not None:
            k = max(1, int(len(self.items) * float(fraction)))
            self.items = random.sample(self.items, k)
        elif max_items is not None and len(self.items) > max_items:
            self.items = random.sample(self.items, max_items)
        # --------------------------------

        # Optional: sanity check
        # from collections import Counter
        # print("class counts:", Counter([y for _, y in self.items]))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        max_retries, tries = 6, 0
        while tries < max_retries:
            off_path, label = self.items[idx]
            try:
                if self.cache and off_path in self._cache:
                    pts = self._cache[off_path]
                else:
                    # sample_points_from_off() already normalizes by default
                    pts = sample_points_from_off(off_path, self.n_points,
                                                 self.method)
                    if self.cache:
                        self._cache[off_path] = pts
                return torch.from_numpy(pts), torch.tensor(label,
                                                           dtype=torch.long)
            except Exception as e:
                if tries == 0:
                    print(f"[warn] Skipping bad OFF: {off_path.name} ({e})")
                tries += 1
                idx = random.randrange(0, len(self.items))
        raise RuntimeError(
            "Too many unreadable meshes in a row; check dataset.")

def collate_fn(batch):
    pts_list, labels = zip(*batch)          # list[(N,3)], list[()]
    pts = torch.stack(pts_list, dim=0)      # (B, N, 3)
    labels = torch.stack(labels, dim=0)     # (B,)
    return pts, labels

# ---- Create datasets with built-in downsizing knobs ----
train_ds = ModelNet40PlaneCarOther(
    ROOT, split="train",
    n_points=N_POINTS, cache=CACHE_IN_RAM, method=SAMPLE_METHOD,
    max_items=MAX_ITEMS,              # or None
    fraction=FRACTION,                # or None
    balance_per_class=BALANCE_PER_CLASS,  # e.g., {0:300, 1:300, 2:300}
)

test_ds = ModelNet40PlaneCarOther(
    ROOT,
    split="test",
    n_points=N_POINTS,
    cache=CACHE_IN_RAM,
    method=SAMPLE_METHOD,
    max_items=MAX_ITEMS,
    fraction=FRACTION,
    balance_per_class=BALANCE_PER_CLASS,
)

# Optional: sanity check
from collections import Counter
print("Class counts:", Counter([y for _, y in train_ds.items]))
print("Class counts:", Counter([y for _, y in test_ds.items]))

train_loader = DataLoader(train_ds,
                          batch_size=BATCH_SIZE,
                          shuffle=True,
                          num_workers=NUM_WORKERS,
                          collate_fn=collate_fn)
test_loader = DataLoader(test_ds,
                         batch_size=BATCH_SIZE,
                         shuffle=False,
                         num_workers=NUM_WORKERS,
                         collate_fn=collate_fn)

print(f"Training data Length: {len(train_ds)}\nTest Data Length: {len(test_ds)}")


In [ ]:
import torch.nn as nn
import torch


class SimplePointNet(nn.Module):
    """
    Per-point MLP (1D convs) → symmetric aggregation (max) → MLP classifier.
    Input:  (B, N, 3)
    Output: (B, NUM_CLASSES) logits
    """

    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.feat = nn.Sequential(
            nn.Conv1d(3, 64, 1),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Conv1d(64, 128, 1),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Conv1d(128, 256, 1),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = x.transpose(1, 2)  # (B, 3, N)
        x = self.feat(x)       # (B, 256, N)
        x = torch.max(x, dim=2).values  # (B, 256)
        return self.classifier(x)       # (B, NUM_CLASSES)


model = SimplePointNet(num_classes=NUM_CLASSES).to(device)
print("Parameter count:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))


In [ ]:
def accuracy_from_logits(logits, y):
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()

def train_one_epoch(model, loader, opt, loss_fn):
    model.train()
    total_loss = total_acc = n = 0
    for pts, y in loader:
        pts, y = pts.to(device), y.to(device).long()
        opt.zero_grad()
        logits = model(pts)
        loss = loss_fn(logits, y)
        loss.backward()
        opt.step()
        bs = pts.size(0)
        total_loss += loss.item() * bs
        total_acc  += accuracy_from_logits(logits, y) * bs
        n += bs
    return total_loss / n, total_acc / n

@torch.no_grad()
def evaluate(model, loader, loss_fn, return_cm: bool = False):
    model.eval()
    total_loss = total_acc = n = 0
    cm = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.int64, device=device)
    for pts, y in loader:
        pts, y = pts.to(device), y.to(device).long()
        logits = model(pts)
        loss = loss_fn(logits, y)
        bs = pts.size(0)
        total_loss += loss.item() * bs
        total_acc  += accuracy_from_logits(logits, y) * bs
        if return_cm:
            preds = logits.argmax(dim=1)
            for t, p in zip(y.view(-1), preds.view(-1)):
                cm[t, p] += 1
        n += bs
    if return_cm:
        return total_loss / n, total_acc / n, cm
    else:
        return total_loss / n, total_acc / n

# 📌 set your custom save path
SAVE_PATH = Path(r"C:\Users\jkarafotis\Desktop\MyDev\Projects\DeepLearning\Models\plane_car_other_pointnet.pth")
SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)  # make sure folder exists

# (optional) class weighting to handle imbalance
from collections import Counter
def _compute_class_weights(ds) -> torch.Tensor:
    counts = Counter([y for _, y in ds.items])
    total = sum(counts.values())
    w = torch.tensor(
        [total / max(1, counts.get(c, 0)) for c in range(NUM_CLASSES)],
        dtype=torch.float32, device=device
    )
    return w / w.mean()

# Use weights if you expect imbalance; otherwise set weight=None
_weights = _compute_class_weights(train_ds)
loss_fn = nn.CrossEntropyLoss(weight=_weights)
opt = torch.optim.Adam(model.parameters(), lr=LR)

# 🔁 Load previous model state if available
best_acc = 0.0
start_epoch = 1

if SAVE_PATH.exists():
    print(f"📂 Found existing model. Loading from {SAVE_PATH} …")
    checkpoint = torch.load(SAVE_PATH, map_location=device)

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
        opt.load_state_dict(checkpoint["optimizer_state_dict"])
        best_acc = checkpoint.get("best_acc", 0.0)
        start_epoch = checkpoint.get("epoch", 0) + 1
        print(f"🔁 Resuming from epoch {start_epoch}, best accuracy: {best_acc:.4f}")
    else:
        model.load_state_dict(checkpoint)
        print("⚠️ Warning: Raw state_dict loaded — no optimizer or epoch info (Creating New Model)")

# 🔁 Continue training from last checkpoint
for epoch in range(start_epoch, start_epoch + EPOCHS):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, opt, loss_fn)
    te_loss, te_acc, te_cm = evaluate(model, test_loader, loss_fn, return_cm=True)

    # 🎯 Per-class accuracy
    with torch.no_grad():
        per_class_den = te_cm.sum(dim=1).clamp(min=1)
        per_class_acc = (te_cm.diag().float() / per_class_den.float()).cpu().tolist()
    pc_str = " | ".join([f"{IDX2NAME[i]}:{a:.3f}" for i, a in enumerate(per_class_acc)])

    print(f"Epoch {epoch:02d} | train {tr_loss:.4f}/{tr_acc:.4f} | test {te_loss:.4f}/{te_acc:.4f} | {pc_str}")

    # 💾 Save model only if accuracy improves
    if te_acc > best_acc:
        best_acc = te_acc
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": opt.state_dict(),
            "best_acc": best_acc
        }, SAVE_PATH)
        print(f"✅ New best model saved at epoch {epoch} with accuracy {best_acc:.4f}")

print(f"🏁 Training complete. Best test accuracy: {best_acc:.4f}")


In [ ]:
# 👁️ Visualize Sample Point-Cloud Predictions (2×3 grid)

import random
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import torch

# Reuse global label map if present
CLASS_NAMES = globals().get("IDX2NAME", {0: "other", 1: "airplane", 2: "car"})

def plot_point_cloud(ax, pts_np, title="", s=3):
    x, y, z = pts_np[:, 0], pts_np[:, 1], pts_np[:, 2]
    ax.scatter(x, y, z, s=s)
    ax.set_title(title, pad=6)
    ax.set_axis_off()
    # equal aspect
    max_range = (np.max(pts_np, axis=0) - np.min(pts_np, axis=0)).max()
    mid = np.mean(pts_np, axis=0)
    for dim, m in zip([ax.set_xlim, ax.set_ylim, ax.set_zlim], mid):
        dim(m - max_range/2, m + max_range/2)

# ---- run one (shuffled) batch for visualization ----
model.eval()

# Make a separate loader just for visualization (doesn't affect your real test_loader)
vis_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,                 # <— key: shuffle so we see a mix of classes
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
)

pts_batch, labels = next(iter(vis_loader))  # pts: (B, N, 3), labels: (B,)
pts_batch, labels = pts_batch.to(device), labels.to(device)

with torch.no_grad():
    preds = model(pts_batch).argmax(1)

# ---- pick ~2 examples per class (pad if missing) ----
num_show = 6
k_per_class = 2
idxs = []
for c in range(NUM_CLASSES):
    cls_idxs = (labels == c).nonzero(as_tuple=True)[0].tolist()
    random.shuffle(cls_idxs)
    idxs += cls_idxs[:k_per_class]

# If the batch didn't contain all classes, pad with randoms
if len(idxs) < num_show:
    pool = [i for i in range(pts_batch.size(0)) if i not in idxs]
    random.shuffle(pool)
    idxs += pool[:num_show - len(idxs)]

fig = plt.figure(figsize=(12, 6))
for i, idx in enumerate(idxs[:num_show], 1):
    ax = fig.add_subplot(2, 3, i, projection='3d')

    pts_np = pts_batch[idx].detach().cpu().numpy()
    true_lbl = labels[idx].item()
    pred_lbl = preds[idx].item()

    # (optional) subsample for speed
    if pts_np.shape[0] > 2048:
        sel = np.random.choice(pts_np.shape[0], 2048, replace=False)
        pts_np = pts_np[sel]

    title_text = f"True: {CLASS_NAMES.get(true_lbl, true_lbl)} | Pred: {CLASS_NAMES.get(pred_lbl, pred_lbl)}"
    color = "green" if true_lbl == pred_lbl else "red"

    plot_point_cloud(ax, pts_np, title=title_text)
    ax.title.set_color(color)

plt.tight_layout()
plt.show()
